# SGD por minibatches

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_optimization/minibatch-sgd.ipynb` · [Lección original](https://d2l.ai/chapter_optimization/minibatch-sgd.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Descenso por gradiente estocástico minibatch
<a id="sec_minibatch_sgd"></a>

Hasta ahora nos encontramos con dos extremos en el enfoque del aprendizaje basado en gradiente: [Referencia sec_gd](https://d2l.ai/chapter_optimization/gd.html#sec-gd) utiliza el conjunto completo de datos para calcular gradientes y actualizar parámetros, uno pasa a la vez. Por el contrario [Referencia sec_sgd](https://d2l.ai/chapter_optimization/sgd.html#sec-sgd) procesa un ejemplo de entrenamiento a la vez para hacer progresos. Cualquiera de ellos tiene sus propios inconvenientes. El descenso por gradiente no es particularmente * eficiente en datos* cuando los datos son muy similares. El descenso por gradiente estocástico no es particularmente * eficiente computacionalmente * ya que las CPUs y GPUs no pueden explotar todo el poder de la vectorización. Esto sugiere que podría haber algo en el medio, y de hecho, eso es lo que hemos estado utilizando hasta ahora en los ejemplos que discutimos.

## Vectorización y Caches
En el centro de la decisión de usar minibatches está la eficiencia computacional. Esto es más fácil de entender cuando se considera la paralelización a múltiples GPUs y múltiples servidores. En este caso necesitamos enviar al menos una imagen a cada GPU. Con 8 GPUs por servidor y 16 servidores ya llegamos a un tamaño minibatch no menor de 128.

Las cosas son un poco más sutiles cuando se trata de GPUs individuales o incluso CPUs. Estos dispositivos tienen múltiples tipos de memoria, a menudo múltiples tipos de unidades computacionales y diferentes restricciones de ancho de banda entre ellos. Por ejemplo, una CPU tiene un pequeño número de registros y luego el L1, L2, y en algunos casos incluso caché L3 (que se comparte entre diferentes núcleos de procesadores). Estas cachés son de tamaño y latencia crecientes (y al mismo tiempo son de ancho de banda decreciente). Basta decir, el procesador es capaz de realizar muchas más operaciones que lo que la interfaz de memoria principal es capaz de proporcionar.

En primer lugar, una CPU de 2GHz con 16 núcleos y AVX-512 vectorización puede procesar hasta $2 \cdot 10^9 \cdot 16 \cdot 32 = 10^{12}$ bytes por segundo. La capacidad de GPUs excede fácilmente este número en un factor de 100. Por otro lado, un procesador de servidor de rango medio puede no tener mucho más de 100 GB/s ancho de banda, es decir, menos de una décima parte de lo que se requeriría para mantener el procesador alimentado. Para empeorar las cosas, no todo el acceso a la memoria se crea igual: las interfaces de memoria son típicamente 64 bit ancho o más ancho (por ejemplo, en GPUs hasta 384 bit), por lo tanto la lectura de un solo byte incurre en el costo de un acceso mucho más amplio.

En segundo lugar, hay una sobrecarga significativa para el primer acceso, mientras que el acceso secuencial es relativamente barato (esto a menudo se llama una lectura de explosión).Hay muchas más cosas a tener en cuenta, tales como caché cuando tenemos múltiples enchufes, chiplets, y otras estructuras. Vea este [Wikipedia article](https://en.wikipedia.org/wiki/Cache_hierarchy) para una discusión más profunda.

La manera de aliviar estas limitaciones es usar una jerarquía de cachés de CPU que son realmente lo suficientemente rápidos para suministrar datos al procesador. Esto es * la fuerza impulsora* detrás de lotinging en el aprendizaje profundo. Para mantener las cosas simples, considere la multiplicación de matriz-matriz, digamos $\mathbf{A} = \mathbf{B}\mathbf{C}$. Tenemos una serie de opciones para calcular $\mathbf{A}$. Por ejemplo, podríamos intentar lo siguiente:

1. Podríamos calcular $\mathbf{A}_{ij} = \mathbf{B}_{i,:} \mathbf{C}_{:,j}$, es decir, podríamos calcular el elemento por medio de productos de punto.
1. Podríamos calcular $\mathbf{A}_{:,j} = \mathbf{B} \mathbf{C}_{:,j}$, es decir, podríamos calcularlo una columna a la vez. Del mismo modo podríamos calcular $\mathbf{A}$ una fila $\mathbf{A}_{i,:}$ a la vez.
1. Simplemente podríamos calcular $\mathbf{A} = \mathbf{B} \mathbf{C}$.
1. Podríamos romper $\mathbf{B}$ y $\mathbf{C}$ en matrices de bloques más pequeñas y calcular $\mathbf{A}$ un bloque a la vez.

Si seguimos la primera opción, tendremos que copiar una fila y un vector de columna en la CPU cada vez que queramos calcular un elemento $\mathbf{A}_{ij}$. Aún peor, debido a que los elementos de la matriz están alineados secuencialmente, se nos requiere así para acceder a muchas ubicaciones desconectadas para uno de los dos vectores como los leemos desde la memoria. La segunda opción es mucho más favorable. En ella, somos capaces de mantener el vector de columna $\mathbf{C}_{:,j}$ en la caché de la CPU mientras seguimos cruzando a través de $\mathbf{B}$. Esto reduce a la mitad el requisito de ancho de banda de memoria con acceso correspondientemente más rápido. Por supuesto, la opción 3 es más deseable. Desafortunadamente, la mayoría de las matrices podrían no encajar completamente en la caché (esto es lo que estamos discutiendo después de todo). Sin embargo, la opción 4 ofrece una alternativa prácticamente útil: podemos mover bloques de la matriz en caché y multiplicarlos localmente. Las bibliotecas optimizadas se encargan de esto para nosotros.

Más allá de la eficiencia computacional, la sobrecarga introducida por Python y por el propio biblioteca de aprendizaje profundo es considerable. Recuerde que cada vez que ejecutamos un comando el intérprete de Python envía un comando al motor MXNet que necesita insertarlo en el grafo computacional y tratar con él durante la programación. Tal sobrecarga puede ser bastante perjudicial. En resumen, es muy recomendable utilizar vectorización (y matrices) siempre que sea posible.


In [ ]:
%matplotlib inline
import time
import numpy as np
import torch
from torch import nn
from laboratorio import d2l

A = torch.zeros(256, 256)
B = torch.randn(256, 256)
C = torch.randn(256, 256)

Puesto que vamos a comparar el tiempo de ejecución con frecuencia en el resto del libro, vamos a definir un temporizador.


In [ ]:
class Timer:  #@save
    """Graba múltiples tiempos de ejecución."""
    def __init__(self):
        self.times = []
        self.start()

    def start(self):
        """Empieza el temporizador."""
        self.tik = time.time()

    def stop(self):
        """Detenga el temporizador y registre la hora en una lista."""
        self.times.append(time.time() - self.tik)
        return self.times[-1]

    def avg(self):
        """Devuelve el tiempo promedio."""
        return sum(self.times) / len(self.times)

    def sum(self):
        """Devuelve la suma de tiempo."""
        return sum(self.times)

    def cumsum(self):
        """Devuelve el tiempo acumulado."""
        return np.array(self.times).cumsum().tolist()

timer = Timer()

La asignación de elementos simplemente itera sobre todas las filas y columnas de $\mathbf{B}$ y $\mathbf{C}$ respectivamente para asignar el valor a $\mathbf{A}$.


In [ ]:
# Computar A = BC un elemento a la vez
timer.start()
for i in range(256):
    for j in range(256):
        A[i, j] = torch.dot(B[i, :], C[:, j])
timer.stop()

Una estrategia más rápida es realizar una asignación en columna.


In [ ]:
# Computar A = BC una columna a la vez
timer.start()
for j in range(256):
    A[:, j] = torch.mv(B, C[:, j])
timer.stop()

Por último, la manera más eficaz es realizar toda la operación en un bloque. Tenga en cuenta que multiplicar cualquier dos matrices $\mathbf{B} \in \mathbb{R}^{m \times n}$ y $\mathbf{C} \in \mathbb{R}^{n \times p}$ toma aproximadamente $2mnp$ operaciones de punto flotante, cuando la multiplicación escalar y la adición se cuentan como operaciones separadas (fundidas en la práctica). Así, multiplicar dos matrices $256 \times 256$ toma $0.03$ mil millones de operaciones de punto flotante. Veamos cuál es la velocidad respectiva de las operaciones.


In [ ]:
# Calcular A = BC de una vez
timer.start()
A = torch.mm(B, C)
timer.stop()

gigaflops = [0.03 / i for i in timer.times]
print(f'performance in Gigaflops: element {gigaflops[0]:.3f}, '
      f'column {gigaflops[1]:.3f}, full {gigaflops[2]:.3f}')

## Minibatches
<a id="sec_minibatches"></a>

En el pasado dimos por sentado que leeríamos *minibatches* de datos en lugar de observaciones únicas para actualizar parámetros. Ahora damos una breve justificación para ello. Procesar observaciones únicas requiere que realicemos muchas multiplicaciones de vectores (o incluso vectores) de matriz única, que es bastante caro y que incurre en un gasto general significativo en nombre del marco subyacente de aprendizaje profundo. Esto se aplica tanto a la evaluación de una red cuando se aplica a los datos (a menudo se refiere como inferencia) y cuando se calculan los gradientes para actualizar parámetros. Es decir, esto se aplica cuando realizamos $\mathbf{w} \leftarrow \mathbf{w} - \eta_t \mathbf{g}_t$ donde

$$\mathbf{g}_t = \partial_{\mathbf{w}} f(\mathbf{x}_{t}, \mathbf{w})$$

Podemos aumentar la eficiencia *computacional* de esta operación aplicándola a un minibatch de observaciones a la vez. Es decir, reemplazamos el gradiente $\mathbf{g}_t$ sobre una sola observación por uno sobre un pequeño lote

$$\mathbf{g}_t = \partial_{\mathbf{w}} \frac{1}{|\mathcal{B}_t|} \sum_{i \in \mathcal{B}_t} f(\mathbf{x}_{i}, \mathbf{w})$$

Veamos lo que esto hace con las propiedades estadísticas de $\mathbf{g}_t$: ya que tanto $\mathbf{x}_t$ como todos los elementos del minibatch $\mathcal{B}_t$ se dibujan uniformemente al azar desde el conjunto de entrenamiento, la expectativa del gradiente permanece inalterada. La varianza, por otro lado, se reduce significativamente. Ya que el gradiente de minibatch se compone de gradientes independientes $b \stackrel{\textrm{def}}{=} |\mathcal{B}_t|$ que se están promediando, su desviación estándar se reduce en un factor de $b^{-\frac{1}{2}}$. Esto, por sí mismo, es una buena cosa, ya que significa que las actualizaciones se alinean de manera más fiable con el gradiente completo.

Nativamente esto indicaría que elegir un gran minibatch $\mathcal{B}_t$ sería universalmente deseable. Desgraciadamente, después de algún punto, la reducción adicional en la desviación estándar es mínima en comparación con el aumento lineal en el costo computacional. En la práctica, escogemos un minibatch que es lo suficientemente grande como para ofrecer una buena eficiencia computacional mientras todavía encaja en la memoria de una GPU. Para ilustrar el ahorro echemos un vistazo a algún código. En él realizamos la misma multiplicación matriz-matriz, pero esta vez se divide en "minibatches" de 64 columnas a la vez.


In [ ]:
timer.start()
for j in range(0, 256, 64):
    A[:, j:j+64] = torch.mm(B, C[:, j:j+64])
timer.stop()
print(f'performance in Gigaflops: block {0.03 / timer.times[3]:.3f}')

Como podemos ver, el cálculo en el minibatch es esencialmente tan eficiente como en la matriz completa. Una palabra de precaución está en orden. En [Referencia sec_batch_norm](https://d2l.ai/chapter_convolutional-modern/batch-norm.html#sec-batch-norm) usamos un tipo de regularización que era muy dependiente de la cantidad de varianza en un minibatch. A medida que aumentamos esta última, la varianza disminuye y con ella el beneficio de la inyección de ruido debido a la normalización por lotes (BatchNorm). Vea por ejemplo, [Ioffe.2017](https://d2l.ai/chapter_references/zreferences.html) para detalles sobre cómo volver a escalar y calcular los términos apropiados.

## Leyendo el conjunto de datos
Echemos un vistazo a cómo se generan los minibatches de manera eficiente a partir de los datos. En el siguiente se utiliza un conjunto de datos desarrollado por la NASA para probar el ala [noise from different aircraft](https://archive.ics.uci.edu/ml/datasets/Airfoil+Self-Noise) para comparar estos algoritmos de optimización. Para comodidad sólo usamos los primeros ejemplos $1,500$. Los datos se blanquean para el preprocesamiento, es decir, eliminamos la media y redimensionamos la varianza a $1$ por coordenada.


In [ ]:
#@save
d2l.DATA_HUB['airfoil'] = (d2l.DATA_URL + 'airfoil_self_noise.dat',
                           '76e5be1548fd8222e5074cf0faae75edff8cf93f')

#@save
def get_data_ch11(batch_size=10, n=1500):
    data = np.genfromtxt(d2l.download('airfoil'),
                         dtype=np.float32, delimiter='\t')
    data = torch.from_numpy((data - data.mean(axis=0)) / data.std(axis=0))
    data_iter = d2l.load_array((data[:n, :-1], data[:n, -1]),
                               batch_size, is_train=True)
    return data_iter, data.shape[1]-1

## Implementación desde cero
Recordemos la implementación de descenso por gradiente estocástico minibatch de [Referencia sec_linear_scratch](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch). A continuación proporcionamos una implementación ligeramente más general. Para mayor comodidad tiene la misma firma de llamada que los otros algoritmos de optimización introducidos más adelante en este capítulo. Específicamente, añadimos la entrada de estado `states` y colocamos el hiperparametro en el diccionario `hyperparams`. Además, promediaremos la pérdida de cada ejemplo de minibatch en la función de entrenamiento, por lo que el gradiente en el algoritmo de optimización no necesita ser dividido por el tamaño del lote.


In [ ]:
def sgd(params, states, hyperparams):
    for p in params:
        p.data.sub_(hyperparams['lr'] * p.grad)
        p.grad.data.zero_()

A continuación, implementamos una función de entrenamiento genérica para facilitar el uso de los otros algoritmos de optimización introducidos más adelante en este capítulo. Inicializa un modelo de regresión lineal y se puede utilizar para entrenar el modelo con descenso por gradiente estocástico minibatch y otros algoritmos introducidos posteriormente.


### Nota docente de Hespérides

Sigue tres objetos diferentes: el valor de la pérdida, su gradiente y la actualización que calcula el optimizador. Comprueba formas y reinicia los gradientes antes de cada paso. En el explorador se mantienen función y punto inicial para comparar trayectorias; una misma tasa no significa el mismo desplazamiento efectivo para todos los métodos.

Vínculo con los apuntes: sesión 2, «SGD por minibatches».


In [ ]:
#@save
def train_ch11(trainer_fn, states, hyperparams, data_iter,
               feature_dim, num_epochs=2):
    # Inicialización
    w = torch.normal(mean=0.0, std=0.01, size=(feature_dim, 1),
                     requires_grad=True)
    b = torch.zeros((1), requires_grad=True)
    net, loss = lambda X: d2l.linreg(X, w, b), d2l.squared_loss
    # Tren
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[0, num_epochs], ylim=[0.22, 0.35])
    n, timer = 0, d2l.Timer()
    for _ in range(num_epochs):
        for X, y in data_iter:
            l = loss(net(X), y).mean()
            l.backward()
            trainer_fn([w, b], states, hyperparams)
            n += X.shape[0]
            if n % 200 == 0:
                timer.stop()
                animator.add(n/X.shape[0]/len(data_iter),
                             (d2l.evaluate_loss(net, data_iter, loss),))
                timer.start()
    print(f'loss: {animator.Y[0][-1]:.3f}, {timer.sum()/num_epochs:.3f} sec/epoch')
    return timer.cumsum(), animator.Y[0]

Vamos a ver cómo la optimización procede para el descenso por gradiente por lotes. Esto se puede lograr estableciendo el tamaño minibatch a 1500 (es decir, al número total de ejemplos). Como resultado, los parámetros del modelo se actualizan sólo una vez por época. Hay poco progreso. De hecho, después de 6 pasos se para el progreso.


In [ ]:
def train_sgd(lr, batch_size, num_epochs=2):
    data_iter, feature_dim = get_data_ch11(batch_size)
    return train_ch11(
        sgd, None, {'lr': lr}, data_iter, feature_dim, num_epochs)

gd_res = train_sgd(1, 1500, 10)

Cuando el tamaño de lote es igual a 1, utilizamos el descenso por gradiente estocástico para la optimización. Para la simplicidad de la implementación hemos escogido una constante (aunque pequeña) tasa de aprendizaje. En el descenso por gradiente estocástico, los parámetros del modelo se actualizan cada vez que se procesa un ejemplo. En nuestro caso esto equivale a 1500 actualizaciones por época. Como podemos ver, la disminución del valor de la función objetiva se ralentiza después de una época. Aunque ambos procedimientos procesan 1500 ejemplos dentro de una época, el descenso por gradiente estocástico consume más tiempo que el descenso por gradiente en nuestro experimento. Esto se debe a que el descenso por gradiente estocástico actualiza los parámetros con más frecuencia y ya que es menos eficiente procesar observaciones individuales una a la vez.


In [ ]:
sgd_res = train_sgd(0.005, 1)

Finalmente, cuando el tamaño de lote es igual a 100, utilizamos el descenso por gradiente estocástico minibatch para la optimización. El tiempo requerido por época es menor que el tiempo necesario para el descenso por gradiente estocástico y el tiempo para el descenso por gradiente por lote.


In [ ]:
mini1_res = train_sgd(.4, 100)

Reducir el tamaño del lote a 10, el tiempo para cada época aumenta porque la carga de trabajo para cada lote es menos eficiente de ejecutar.


In [ ]:
mini2_res = train_sgd(.05, 10)

Ahora podemos comparar el tiempo vs. pérdida para los cuatro experimentos anteriores. Como se puede ver, aunque el descenso por gradiente estocástico converge más rápido que GD en términos de número de ejemplos procesados, utiliza más tiempo para alcanzar la misma pérdida que GD porque calcular el ejemplo de gradiente por ejemplo no es tan eficiente. Minibatch descenso por gradiente estocástico es capaz de compensar la velocidad de convergencia y eficiencia de cálculo. Un tamaño de minibatch de 10 es más eficiente que descenso por gradiente estocástico; un tamaño de minibatch de 100 incluso supera a GD en términos de tiempo de ejecución.


In [ ]:
d2l.set_figsize([6, 3])
d2l.plot(*list(map(list, zip(gd_res, sgd_res, mini1_res, mini2_res))),
         'tiempo (s)', 'loss', xlim=[1e-2, 10],
         legend=['gd', 'sgd', 'batch size=100', 'batch size=10'])
d2l.plt.gca().set_xscale('log')

## Implementación concisa
En Gluon, podemos usar la clase `Trainer` para llamar algoritmos de optimización. Esto se utiliza para implementar una función de entrenamiento genérica. Usaremos esto a lo largo del capítulo actual.


In [ ]:
#@save
def train_concise_ch11(trainer_fn, hyperparams, data_iter, num_epochs=4):
    # Inicialización
    net = nn.Sequential(nn.Linear(5, 1))
    def init_weights(module):
        if type(module) == nn.Linear:
            torch.nn.init.normal_(module.weight, std=0.01)
    net.apply(init_weights)

    optimizer = trainer_fn(net.parameters(), **hyperparams)
    loss = nn.MSELoss(reduction='none')
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[0, num_epochs], ylim=[0.22, 0.35])
    n, timer = 0, d2l.Timer()
    for _ in range(num_epochs):
        for X, y in data_iter:
            optimizer.zero_grad()
            out = net(X)
            y = y.reshape(out.shape)
            l = loss(out, y)
            l.mean().backward()
            optimizer.step()
            n += X.shape[0]
            if n % 200 == 0:
                timer.stop()
                # `MSELoss` calcula el error al cuadrado sin el factor 1/2
                animator.add(n/X.shape[0]/len(data_iter),
                             (d2l.evaluate_loss(net, data_iter, loss) / 2,))
                timer.start()
    print(f'loss: {animator.Y[0][-1]:.3f}, {timer.sum()/num_epochs:.3f} sec/epoch')

El uso de Gluon para repetir el último experimento muestra un comportamiento idéntico.


In [ ]:
data_iter, _ = get_data_ch11(10)
trainer = torch.optim.SGD
train_concise_ch11(trainer, {'lr': 0.01}, data_iter)

## Resumen
* La Vectorización hace que el código sea más eficiente debido a la reducción de gastos derivados del biblioteca de aprendizaje profundo y debido a una mejor ubicación de memoria y almacenamiento en caché en CPUs y GPUs.
* Existe una compensación entre la eficiencia estadística derivada del descenso por gradiente estocástico y la eficiencia computacional derivada del procesamiento de grandes lotes de datos a la vez.
* El descenso por gradiente estocástico minibatch ofrece lo mejor de ambos mundos: eficiencia computacional y estadística.
* En descenso por gradiente estocástico minibatch procesamos lotes de datos obtenidos por una permutación aleatoria de los datos de entrenamiento (es decir, cada observación se procesa sólo una vez por época, aunque en orden aleatorio).
* Es aconsejable disminuir las tasas de aprendizaje durante la formación.
* En general, el descenso por gradiente estocástico es más rápido que el descenso por gradiente estocástico y el descenso por gradiente para convergencia a un riesgo menor, cuando se mide en términos de tiempo de reloj.

## Ejercicios
1. Modificar el tamaño del lote y la tasa de aprendizaje y observar la tasa de disminución por el valor de la función objetiva y el tiempo consumido en cada época.
1. Lea la documentación de MXNet y utilice la función `Trainer` clase `set_learning_rate` para reducir la tasa de aprendizaje del descenso por gradiente estocástico minibatch a 1/10 de su valor previo después de cada época.
1. Comparar descenso por gradiente estocástico minibatch con una variante que en realidad *muestra con reemplazo* del conjunto de entrenamiento. ¿Qué sucede?
1. Un genio malvado replica su conjunto de datos sin decirle (es decir, cada observación ocurre dos veces y su conjunto de datos crece hasta el doble de su tamaño original, pero nadie le dijo). ¿Cómo cambia el comportamiento de descenso por gradiente estocástico, descenso por gradiente estocástico minibatch y el de descenso por gradiente?


[Debate del original](https://discuss.d2l.ai/t/1068)
